In [1]:
import os

# Affiche tous les fichiers disponibles dans /kaggle/input, avec leur taille
for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
    for nom in fichiers:
        chemin = os.path.join(dossier, nom)
        taille_go = os.path.getsize(chemin) / 1e9
        print(chemin, "->", round(taille_go, 2), "Go")

/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json -> 5.67 Go


In [2]:
import json

chemin = "/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"

# Le fichier contient un article par ligne, au format JSON.
# On le lit ligne par ligne : il est trop gros (5,67 Go) pour être chargé en entier.
with open(chemin, "r") as f:
    for i, ligne in enumerate(f):
        article = json.loads(ligne)
        if i == 0:
            print("Champs disponibles :", list(article.keys()))
            print("=" * 80)
        print("id         :", article["id"])
        print("categories :", article["categories"])
        print("title      :", article["title"])
        print("abstract   :", article["abstract"][:300], "...")
        print("-" * 80)
        if i == 2:  # on s'arrête après 3 articles
            break

Champs disponibles : ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']
id         : 0704.0001
categories : hep-ph
title      : Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies
abstract   :   A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
a ...
--------------------------------------------------------------------------------
id         : 0704.0002
categories : math.CO cs.CG
title      : Sparsity-certifying Graph Decompositions
abstract   :   We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use
it obtain a characterization of the family of $(k,\ell)$-sparse graphs 

In [3]:
import json
from collections import Counter
from tqdm.auto import tqdm

chemin = "/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"

compteur_archives = Counter()
nb_articles = 0

with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Lecture des articles"):
        article = json.loads(ligne)
        nb_articles += 1
        # "math.CO cs.CG" -> ["math.CO", "cs.CG"] -> {"math", "cs"}
        archives = {cat.split(".")[0] for cat in article["categories"].split()}
        compteur_archives.update(archives)

print("Nombre total d'articles      :", nb_articles)
print("Nombre d'archives différentes :", len(compteur_archives))
print()
for archive, n in compteur_archives.most_common():
    print(f"{archive:15s} {n}")

Lecture des articles: 0it [00:00, ?it/s]

Nombre total d'articles      : 3233620
Nombre d'archives différentes : 38

cs              1045061
math            799814
cond-mat        432326
astro-ph        396081
physics         318641
hep-ph          200767
quant-ph        190019
hep-th          187217
stat            153817
eess            142427
gr-qc           126032
math-ph         93799
nucl-th         64007
hep-ex          61926
q-bio           57384
nlin            48203
hep-lat         30966
nucl-ex         29152
q-fin           26461
econ            17990
chao-dyn        2398
q-alg           1578
alg-geom        1423
solv-int        1413
cmp-lg          894
dg-ga           732
patt-sol        650
adap-org        584
funct-an        427
mtrl-th         262
chem-ph         251
comp-gas        221
supr-con        175
atom-ph         123
acc-phys        49
plasm-ph        38
ao-sci          17
bayes-an        16


In [4]:
# Pour chaque groupe : la liste des archives arXiv qui en font partie
ARCHIVES_PAR_GROUPE = {
    "cs":      ["cs", "cmp-lg"],
    "math":    ["math", "q-alg", "alg-geom", "dg-ga", "funct-an"],
    "stat":    ["stat"],
    "eess":    ["eess"],
    "q-bio":   ["q-bio"],
    "q-fin":   ["q-fin"],
    "econ":    ["econ"],
    "physics": ["physics", "cond-mat", "astro-ph", "hep-ph", "hep-th", "hep-ex",
                "hep-lat", "quant-ph", "gr-qc", "math-ph", "nucl-th", "nucl-ex",
                "nlin", "chao-dyn", "solv-int", "patt-sol", "adap-org", "comp-gas",
                "mtrl-th", "supr-con", "chem-ph", "atom-ph", "acc-phys",
                "plasm-ph", "ao-sci", "bayes-an"],
}

# On retourne le tableau pour pouvoir chercher dans l'autre sens : archive -> groupe
GROUPE_PAR_ARCHIVE = {}
for groupe, archives in ARCHIVES_PAR_GROUPE.items():
    for archive in archives:
        GROUPE_PAR_ARCHIVE[archive] = groupe


def categories_vers_groupes(categories):
    """Exemple : 'math.CO cs.CG' -> {'math', 'cs'}"""
    archives = {cat.split(".")[0] for cat in categories.split()}
    return {GROUPE_PAR_ARCHIVE[a] for a in archives}


# Vérification : est-ce qu'on a oublié une des 38 archives ?
oubliees = set(compteur_archives) - set(GROUPE_PAR_ARCHIVE)
print("Archives oubliées :", oubliees if oubliees else "aucune")

# Tests
print(categories_vers_groupes("hep-ph"))
print(categories_vers_groupes("math.CO cs.CG"))
print(categories_vers_groupes("cs.LG stat.ML"))

Archives oubliées : aucune
{'physics'}
{'math', 'cs'}
{'stat', 'cs'}


In [5]:
compteur_groupes = Counter()     # combien d'articles dans chaque groupe
compteur_nb_groupes = Counter()  # combien d'articles ont 1 groupe, 2 groupes, 3 groupes...

with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Lecture des articles"):
        article = json.loads(ligne)
        groupes = categories_vers_groupes(article["categories"])  # ta fonction de la cellule 4
        compteur_groupes.update(groupes)          # +1 pour chaque groupe de l'article
        compteur_nb_groupes[len(groupes)] += 1    # +1 dans la case "1 groupe", "2 groupes"...

print("Nombre d'articles par groupe :")
for groupe, n in compteur_groupes.most_common():
    print(f"  {groupe:8s} {n:>10,}")

print()
print("Nombre de groupes par article :")
for k in sorted(compteur_nb_groupes):
    print(f"  {k} groupe(s) : {compteur_nb_groupes[k]:>10,} articles")

Lecture des articles: 0it [00:00, ?it/s]

Nombre d'articles par groupe :
  physics   1,624,779
  cs        1,045,061
  math        799,815
  stat        153,817
  eess        142,427
  q-bio        57,384
  q-fin        26,461
  econ         17,990

Nombre de groupes par article :
  1 groupe(s) :  2,682,758 articles
  2 groupe(s) :    473,437 articles
  3 groupe(s) :     71,870 articles
  4 groupe(s) :      5,289 articles
  5 groupe(s) :        260 articles
  6 groupe(s) :          6 articles


In [6]:
import random
import pandas as pd

random.seed(42)  # pour obtenir le même tirage à chaque exécution

GROUPES = ["cs", "math", "physics", "q-bio", "q-fin", "stat", "eess", "econ"]
OBJECTIF_PAR_GROUPE = 2500

# Taux de tirage de chaque groupe (calculé avec les chiffres de la cellule 5)
taux = {g: min(1.0, OBJECTIF_PAR_GROUPE / compteur_groupes[g]) for g in GROUPES}
print("Taux de tirage :")
for g in GROUPES:
    print(f"  {g:8s} {100 * taux[g]:.2f} %")

echantillon = []

with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Tirage"):
        article = json.loads(ligne)
        groupes = categories_vers_groupes(article["categories"])

        # Taux de l'article = le plus élevé parmi ses groupes
        taux_article = max(taux[g] for g in groupes)
        if random.random() >= taux_article:
            continue  # pas tiré -> on passe à l'article suivant

        # Résumé trop court (souvent "This paper has been withdrawn...") -> on l'ignore
        if len(article["abstract"].split()) < 30:
            continue

        # Une ligne du tableau final : id, titre, résumé + 8 colonnes à 0 ou 1
        resultat = {"id": article["id"], "title": article["title"], "abstract": article["abstract"]}
        for g in GROUPES:
            resultat[g] = 1 if g in groupes else 0
        echantillon.append(resultat)

df = pd.DataFrame(echantillon)
print("\nNombre d'articles tirés :", len(df))
print("\nArticles par groupe dans l'échantillon :")
print(df[GROUPES].sum().sort_values(ascending=False))

df.to_csv("/kaggle/working/arxiv_echantillon.csv", index=False)
print("\nFichier sauvegardé : /kaggle/working/arxiv_echantillon.csv")

Taux de tirage :
  cs       0.24 %
  math     0.31 %
  physics  0.15 %
  q-bio    4.36 %
  q-fin    9.45 %
  stat     1.63 %
  eess     1.76 %
  econ     13.90 %


Tirage: 0it [00:00, ?it/s]


Nombre d'articles tirés : 17349

Articles par groupe dans l'échantillon :
cs         6819
physics    4154
math       4040
stat       3229
q-fin      2910
q-bio      2592
eess       2558
econ       2440
dtype: int64

Fichier sauvegardé : /kaggle/working/arxiv_echantillon.csv
